# Kvasir-SEG Polyp — Boundary Loss Ablation

비교: `ce_dice` / `plwce_dice` (baseline) vs `ce_dice_boundary` / `plwce_dice_boundary` / `plwce_boundary`

In [ ]:
# === Cell 0: 환경 설정 ===
import subprocess, sys
for pkg in ['openpyxl', 'kagglehub', 'scipy', 'optuna', 'albumentations']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
import os, warnings, json, random, glob, urllib.request
warnings.filterwarnings('ignore')
os.environ['TQDM_DISABLE'] = '1'
import numpy as np, cv2
from tqdm import tqdm
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix
sys.path.insert(0, '/root/imbalanced-data-LWCE/medical_data')
from custom_losses import get_loss_function

DOMAIN = 'kvasir'; NUM_CLASSES = 2; CLASS_NAMES = ['Background', 'Polyp']
IMG_SIZE = 352; BATCH_SIZE = 8; NUM_WORKERS = 0; SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
RESULTS_DIR = '/root/imbalanced-data-LWCE/medical_data/boundary_ablation/results/kvasir'
os.makedirs(RESULTS_DIR, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}'); print('환경 설정 완료')

In [ ]:
# === Cell 1: 데이터 로드 ===
import kagglehub
_path = kagglehub.dataset_download('debeshjha1/kvasirseg')
DATA_DIR = os.path.join(_path, 'Kvasir-SEG', 'Kvasir-SEG')
if not os.path.exists(DATA_DIR):
    DATA_DIR = _path
print('Kvasir path:', DATA_DIR)

all_images = sorted(glob.glob(os.path.join(DATA_DIR, 'images', '*.jpg')))
all_masks  = sorted(glob.glob(os.path.join(DATA_DIR, 'masks',  '*.jpg')))
print(f'Images: {len(all_images)}  Masks: {len(all_masks)}')

tr_imgs, tmp_imgs, tr_masks, tmp_masks = train_test_split(
    all_images, all_masks, test_size=0.2, random_state=SEED)
val_imgs, test_imgs, val_masks, test_masks = train_test_split(
    tmp_imgs, tmp_masks, test_size=0.5, random_state=SEED)
print(f'Train: {len(tr_imgs)}  Val: {len(val_imgs)}  Test: {len(test_imgs)}')

class PolypDataset(Dataset):
    def __init__(self, imgs, masks, transform=None):
        self.imgs = sorted(imgs); self.masks = sorted(masks); self.transform = transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        img  = cv2.cvtColor(cv2.imread(self.imgs[i]),  cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.masks[i], cv2.IMREAD_GRAYSCALE)
        _, mask = cv2.threshold(mask, 127, 1, cv2.THRESH_BINARY)
        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']
        return img, mask.long()

train_tf = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.HorizontalFlip(p=0.5),
                       A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
                       A.Normalize(), ToTensorV2()])
val_tf   = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(), ToTensorV2()])

train_loader = DataLoader(PolypDataset(tr_imgs, tr_masks, train_tf),
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(PolypDataset(val_imgs, val_masks, val_tf),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(PolypDataset(test_imgs, test_masks, val_tf),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print('클래스 비율 계산 중...')
bg, fg = 0, 0
for mp in tr_masks:
    m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
    fg += int((m > 127).sum()); bg += int((m <= 127).sum())
class_counts = [bg, fg]
print(f'Train: {len(tr_imgs)} | Val: {len(val_imgs)} | Test: {len(test_imgs)}')
print(f'BG: {bg:,}  FG(polyp): {fg:,}  Ratio: {bg/fg:.1f}:1')
print(f'class_counts = {class_counts}')
print('DataLoader 완료')

In [ ]:
# === Cell 2: 모델 정의 ===

# PraNet 설정
pranet_lib = '/tmp/PraNet/lib'
if pranet_lib not in sys.path:
    sys.path.insert(0, pranet_lib)

if not os.path.exists('/tmp/PraNet'):
    os.system('git clone https://github.com/DengPingFan/PraNet.git /tmp/PraNet')
    target = '/tmp/PraNet/lib/PraNet_Res2Net.py'
    with open(target) as f: code = f.read()
    if 'from .Res2Net_v1b' in code:
        with open(target, 'w') as f: f.write(code.replace('from .Res2Net_v1b', 'from Res2Net_v1b'))

wp = '/tmp/PraNet/models/res2net50_v1b_26w_4s-3cf99910.pth'
os.makedirs('/tmp/PraNet/models', exist_ok=True)
if not os.path.exists(wp):
    print('Res2Net 가중치 다운로드 중...')
    urllib.request.urlretrieve(
        'https://shanghuagao.oss-cn-beijing.aliyuncs.com/res2net/res2net50_v1b_26w_4s-3cf99910.pth', wp)

r2n = '/tmp/PraNet/lib/Res2Net_v1b.py'
with open(r2n) as f: code = f.read()
hc = '/media/nercms/NERCMS/GepengJi/Medical_Seqmentation/CRANet/models/res2net50_v1b_26w_4s-3cf99910.pth'
if hc in code:
    with open(r2n, 'w') as f: f.write(code.replace(hc, wp))

for k in [k for k in sys.modules if 'Res2Net' in k or 'PraNet_Res2Net' in k]:
    del sys.modules[k]
from PraNet_Res2Net import PraNet

def to_2ch_logits(p):
    return torch.cat([-p, p], dim=1)

def build_model():
    return PraNet().to(device)

def compute_val_dice(model, loader):
    model.eval()
    tp = fp = fn = 0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            _, _, _, res = model(imgs)
            res = F.interpolate(res, size=masks.shape[1:], mode='bilinear', align_corners=True)
            prob = torch.sigmoid(res).squeeze(1)
            pred = (prob > 0.5).long()
            tp += ((pred == 1) & (masks == 1)).sum().item()
            fp += ((pred == 1) & (masks == 0)).sum().item()
            fn += ((pred == 0) & (masks == 1)).sum().item()
    return float(2 * tp / (2 * tp + fp + fn + 1e-8))

def compute_val_metrics(model, loader):
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(device)
            _, _, _, res = model(imgs)
            res = F.interpolate(res, size=masks.shape[1:], mode='bilinear', align_corners=True)
            prob = torch.sigmoid(res).squeeze(1).cpu().numpy()
            pred = (prob > 0.5).astype(np.int64)
            all_probs.append(prob.flatten())
            all_preds.append(pred.flatten())
            all_labels.append(masks.numpy().flatten())
    all_probs  = np.concatenate(all_probs)
    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    TP = ((all_preds == 1) & (all_labels == 1)).sum()
    FP = ((all_preds == 1) & (all_labels == 0)).sum()
    TN = ((all_preds == 0) & (all_labels == 0)).sum()
    FN = ((all_preds == 0) & (all_labels == 1)).sum()
    dice = 2 * TP / (2 * TP + FP + FN + 1e-8)
    sens = TP / (TP + FN + 1e-8)
    spec = TN / (TN + FP + 1e-8)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except Exception:
        auc = 0.0
    return {'Dice': float(dice), 'Sensitivity': float(sens), 'Specificity': float(spec), 'AUC': float(auc)}

print('모델 + 유틸리티 함수 준비 완료')

In [ ]:
# === Cell 3: 학습 함수 ===

FINAL_EPOCHS = 20
FINAL_LR     = 1e-4

def train_model(loss_name, alpha=1.0, gamma=2.0, epochs=20, lr=1e-4,
                subset_ratio=1.0, tag=''):
    model     = build_model()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = get_loss_function(loss_name, class_counts=class_counts, alpha=alpha, gamma=gamma)
    name = f'{loss_name}_a{alpha:.2f}' if alpha != 1.0 else loss_name
    if tag: name = f'{tag}_{name}'
    print(f"\n{'='*60}\n{name}  (epochs={epochs})\n{'='*60}")
    if subset_ratio < 1.0:
        n = max(1, int(len(train_loader.dataset) * subset_ratio))
        sub_ds = torch.utils.data.Subset(
            train_loader.dataset, random.sample(range(len(train_loader.dataset)), n))
        loader = DataLoader(sub_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    else:
        loader = train_loader
    history = {'loss': [], 'val_dice': []}
    best_dice = 0.0
    save_path = f'/tmp/ba_kvasir_{name}.pth'
    for epoch in range(epochs):
        # BL annealing: set ONCE per epoch before inner loop
        if criterion.boundary_loss is not None:
            alpha_t = min(epoch / epochs, 0.5)
            criterion.set_boundary_alpha(alpha_t)
        model.train()
        epoch_loss = 0.0
        for imgs, masks in tqdm(loader, desc=f'Ep{epoch+1:02d}/{epochs}', leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            o5, o4, o3, o2 = model(imgs)
            loss = 0
            for out in [o5, o4, o3, o2]:
                out = F.interpolate(out, size=masks.shape[1:], mode='bilinear', align_corners=True)
                loss = loss + criterion(to_2ch_logits(out), masks)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()
        avg_loss = epoch_loss / len(loader)
        val_dice = compute_val_dice(model, val_loader)
        history['loss'].append(avg_loss)
        history['val_dice'].append(val_dice)
        print(f'Ep{epoch+1:02d} | Loss: {avg_loss:.4f} | Val Dice: {val_dice:.4f}', end='')
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), save_path)
            print('  <- Best!', end='')
        print()
    model.load_state_dict(torch.load(save_path, weights_only=True))
    print(f'최고 Val Dice: {best_dice:.4f}')
    return model, history, best_dice

print('train_model() 준비 완료')

In [ ]:
# === Cell 4: Optuna — PLWCE alpha 탐색 (Boundary Loss 환경) ===
import optuna, traceback
optuna.logging.set_verbosity(optuna.logging.WARNING)
os.environ['TQDM_DISABLE'] = '1'

ALPHA_LOW    = 2.5
ALPHA_HIGH   = 15.0
PROXY_EPOCHS = 8
PROXY_RATIO  = 0.15
N_TRIALS     = 20

def make_objective(loss_name):
    def objective(trial):
        alpha = trial.suggest_float('alpha', ALPHA_LOW, ALPHA_HIGH)
        try:
            _, _, dice = train_model(loss_name, alpha=alpha,
                                     epochs=PROXY_EPOCHS, subset_ratio=PROXY_RATIO)
            return dice
        except Exception:
            traceback.print_exc()
            return None
    return objective

# --- plwce_dice_boundary ---
print('Optuna: plwce_dice_boundary ...')
sampler_pdb = optuna.samplers.GridSampler(
    {'alpha': np.linspace(ALPHA_LOW, ALPHA_HIGH, N_TRIALS).tolist()}
)
study_pdb = optuna.create_study(direction='maximize', sampler=sampler_pdb)
study_pdb.optimize(make_objective('plwce_dice_boundary'), n_trials=N_TRIALS)
best_trials_pdb = [t for t in study_pdb.trials if t.value is not None]
best_alpha_with_dice = (
    best_trials_pdb[int(np.argmax([t.value for t in best_trials_pdb]))].params['alpha']
    if best_trials_pdb else 7.0
)
print(f'  best alpha (with Dice): {best_alpha_with_dice:.3f}')

# --- plwce_boundary (no Dice) ---
print('Optuna: plwce_boundary (no Dice) ...')
sampler_pb = optuna.samplers.GridSampler(
    {'alpha': np.linspace(ALPHA_LOW, ALPHA_HIGH, N_TRIALS).tolist()}
)
study_pb = optuna.create_study(direction='maximize', sampler=sampler_pb)
study_pb.optimize(make_objective('plwce_boundary'), n_trials=N_TRIALS)
best_trials_pb = [t for t in study_pb.trials if t.value is not None]
best_alpha_without_dice = (
    best_trials_pb[int(np.argmax([t.value for t in best_trials_pb]))].params['alpha']
    if best_trials_pb else 5.0
)
print(f'  best alpha (no Dice):   {best_alpha_without_dice:.3f}')

optuna_data = {
    'plwce_dice_boundary': {'best_alpha': best_alpha_with_dice},
    'plwce_boundary':      {'best_alpha': best_alpha_without_dice},
}
with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_boundary_optuna.json'), 'w') as f:
    json.dump(optuna_data, f, indent=2)
print('Optuna 결과 저장 완료')

In [ ]:
# === Cell 5: Boundary Ablation 전체 학습 ===

# --- Optuna 결과 로드 (Cell 4 미실행 시 JSON fallback) ---
try:
    _ = best_alpha_with_dice
except NameError:
    try:
        with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_boundary_optuna.json')) as f:
            d = json.load(f)
        best_alpha_with_dice    = d['plwce_dice_boundary']['best_alpha']
        best_alpha_without_dice = d['plwce_boundary']['best_alpha']
        print(f'Optuna 로드: with_dice={best_alpha_with_dice:.3f}, no_dice={best_alpha_without_dice:.3f}')
    except FileNotFoundError:
        best_alpha_with_dice    = 7.0
        best_alpha_without_dice = 5.0
        print(f'Optuna 미실행 → fallback alpha with_dice=7.0, no_dice=5.0')

experiments = [
    ('ce_dice',             1.0,                     'CE+Dice                     [baseline]'),
    ('plwce_dice',          best_alpha_with_dice,    f'PLWCE+Dice                  (α={best_alpha_with_dice:.3f}) [baseline]'),
    ('ce_dice_boundary',    1.0,                     'CE+Dice+BL                  [literature]'),
    ('plwce_dice_boundary', best_alpha_with_dice,    f'PLWCE+Dice+BL               (α={best_alpha_with_dice:.3f})'),
    ('plwce_boundary',      best_alpha_without_dice, f'PLWCE+BL     (no Dice)       (α={best_alpha_without_dice:.3f})'),
]

all_results = {}
for loss_name, alpha, label in experiments:
    model, history, best_dice = train_model(
        loss_name=loss_name, alpha=alpha,
        epochs=FINAL_EPOCHS, lr=FINAL_LR, tag='ba')
    all_results[label] = {
        'model': model, 'history': history, 'best_dice': best_dice,
        'loss_name': loss_name, 'alpha': alpha,
    }

print('\n' + '='*65)
print('[Boundary Ablation 요약 — Val Dice]')
print(f"{'Loss':<52} {'Val Dice':>9}")
print('-'*63)
for label, v in all_results.items():
    print(f"{label:<52} {v['best_dice']:>9.4f}")

In [ ]:
# === Cell 6: 평가 및 결과 저장 ===
import pandas as pd

COLORS = ['#4878D0', '#EE854A', '#6ACC65', '#D65F5F', '#B47CC7']

# --- 학습 곡선 ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
for i, (label, v) in enumerate(all_results.items()):
    c = COLORS[i % len(COLORS)]
    ax1.plot(v['history']['loss'],     label=label[:40], color=c)
    ax2.plot(v['history']['val_dice'], label=label[:40], color=c)
ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch'); ax1.legend(fontsize=7)
ax2.set_title('Val Dice');      ax2.set_xlabel('Epoch'); ax2.legend(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'boundary_ablation_curves.png'), dpi=150)
plt.show()

# --- Test set 정량 평가 ---
print('\n[Test Set 정량 평가]')
print(f"{'Loss':<52} {'Dice':>7} {'Sens':>7} {'Spec':>7} {'AUC':>7}")
print('-' * 82)

final_results = {}
for label, v in all_results.items():
    m = compute_val_metrics(v['model'], test_loader)
    final_results[label] = m
    print(f"{label:<52} {m['Dice']:>7.4f} {m['Sensitivity']:>7.4f} {m['Specificity']:>7.4f} {m['AUC']:>7.4f}")

# --- 바 차트 ---
labels = list(final_results.keys())
dices  = [final_results[l]['Dice'] for l in labels]
idx    = sorted(range(len(dices)), key=lambda i: dices[i], reverse=True)
fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(range(len(labels)), [dices[i] for i in idx],
              color=[COLORS[i % len(COLORS)] for i in range(len(labels))])
ax.set_xticks(range(len(labels)))
ax.set_xticklabels([labels[i][:40] for i in idx], rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Test Dice')
ax.set_title(f'Boundary Ablation — Test Dice ({DOMAIN.upper()})')
for bar, val in zip(bars, [dices[i] for i in idx]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'boundary_ablation_bar.png'), dpi=150)
plt.show()

# --- JSON 저장 ---
with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_boundary_ablation.json'), 'w') as f:
    json.dump({l: {k: v for k, v in m.items()} for l, m in final_results.items()},
              f, indent=2, ensure_ascii=False)
print(f'결과 저장 완료: {RESULTS_DIR}')

# --- Excel 저장 ---
summary_rows = []
for label, m in final_results.items():
    summary_rows.append({
        'Loss_Function': label,
        'Dice':          round(m['Dice'],        4),
        'Sensitivity':   round(m['Sensitivity'], 4),
        'Specificity':   round(m['Specificity'], 4),
        'AUC':           round(m['AUC'],         4),
    })
df_summary = pd.DataFrame(summary_rows)

history_rows = []
for label, v in all_results.items():
    for ep, (loss, dice) in enumerate(
            zip(v['history']['loss'], v['history']['val_dice']), 1):
        history_rows.append({
            'Loss_Function': label,
            'Epoch':         ep,
            'Train_Loss':    round(loss, 6),
            'Val_Dice':      round(dice, 6),
        })
df_history = pd.DataFrame(history_rows)

excel_path = os.path.join(RESULTS_DIR, f'{DOMAIN}_boundary_ablation.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_summary.to_excel(writer, sheet_name='Summary',          index=False)
    df_history.to_excel(writer, sheet_name='Training_History', index=False)
print(f'Excel 저장: {excel_path}')